In [ ]:
# @title **Install dependencies (portable & non-fatal)**
# ---------------------------------------------------------------------------
# Installs into the CURRENT kernel via `sys.executable -m pip` so it targets the
# right Python in Colab AND in local venvs (a bare `!pip` can hit the wrong one).
# Wrapped in try/except so that if the machine is offline or a wheel fails, the
# notebook does NOT halt: the guarded imports in the next cell let it keep running
# against the cached dataset. `google-serp-api` is dropped (never imported);
# `python-dotenv` is added for local `.env` credential loading.
# ---------------------------------------------------------------------------
import sys, subprocess

PACKAGES = [
    "smolagents[toolkit]",   # multi-agent framework
    "google-search-results",  # provides `serpapi.GoogleSearch`
    "python-dotenv",          # local .env credential loading
]

try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PACKAGES],
        check=True, timeout=600,
    )
    print("[info] Dependencies installed into the active kernel.")
except Exception as exc:
    print(f"[info] Skipping install ({exc.__class__.__name__}); continuing with "
          "already-available packages (guarded imports handle any that are missing).")


**1. Setup and Agent Definition**


In [ ]:
# @title **LLM Setup (portable: Google Colab, local, or fully offline)**
# ---------------------------------------------------------------------------
# This cell was modified to run in three environments without edits:
#   1. Google Colab  -> reads secrets from google.colab.userdata
#   2. Local Jupyter -> reads secrets from environment vars / a local .env file
#   3. Offline/no-keys -> everything still runs against the cached dataset below
#
# CREDENTIAL SAFETY: no keys are ever printed. We only report set/missing.
# ---------------------------------------------------------------------------
import os
from typing import Dict, Any

# Master switch. Leave False to run reproducibly offline with cached data.
# Set True ONLY when you have live SerpAPI + LLM keys and want fresh searches.
RUN_LIVE = False


def _load_secret(name: str):
    """Load a credential from Colab userdata, then env vars / .env, else None."""
    # 1) Google Colab secrets manager
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    # 2) Local: optionally hydrate os.environ from a .env file, then read it
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass
    return os.environ.get(name)


serp_api = _load_secret("serp_api")
GL_OpenAI = _load_secret("GL_OpenAI")

# smolagents / serpapi are OPTIONAL offline. Guard the imports so the whole
# notebook still executes top-to-bottom when they (or the keys) are missing.
try:
    from smolagents import OpenAIServerModel, CodeAgent, WebSearchTool, Tool, tool
    SMOLAGENTS_AVAILABLE = True
except Exception as exc:  # pragma: no cover - environment dependent
    print(f"[info] smolagents unavailable ({exc.__class__.__name__}); offline mode.")
    SMOLAGENTS_AVAILABLE = False

    # No-op stand-ins so the later @tool-decorated cells still define plain
    # Python functions we can call directly against the cached dataset.
    def tool(fn=None, **kwargs):
        return fn if callable(fn) else (lambda f: f)

    class _OfflineStub:
        def __init__(self, *a, **k):
            pass

        def run(self, *a, **k):
            raise RuntimeError("Live agents are unavailable offline; use cached results.")

    OpenAIServerModel = CodeAgent = WebSearchTool = Tool = _OfflineStub

try:
    from serpapi import GoogleSearch
except Exception:
    GoogleSearch = None  # only needed for live search tools

# Build a live LLM client only if we can AND want to.
model = None
if RUN_LIVE and SMOLAGENTS_AVAILABLE and GL_OpenAI:
    model = OpenAIServerModel(
        model_id="gpt-4o-mini",
        api_base="https://aibe.mygreatlearning.com/openai/v1",
        api_key=GL_OpenAI,
    )
    print("[info] Live model initialized.")
else:
    print(
        "[info] Offline mode -> "
        f"RUN_LIVE={RUN_LIVE}, smolagents={SMOLAGENTS_AVAILABLE}, "
        f"serp_api={'set' if serp_api else 'missing'}, "
        f"GL_OpenAI={'set' if GL_OpenAI else 'missing'}"
    )


## **Tools**

### **Define - Flight Tool**

In [ ]:
flight_system_prompt = """
You are a Travel Assistant Agent responsible for searching flight details between origin and destination locations for a given date and trip type.

For the flight search, you must use the `search_flights` tool with arguments:
- origin: airport code or city code where the flight departs from (e.g., 'CDG')
- destination: airport code or city code where the flight arrives (e.g., 'BER')
- date: departure date in YYYY-MM-DD format (e.g., '2025-10-10')
- trip_type: integer trip type (1=Round trip, 2=One way, 3=Multi-city)

Return the flight info formatted exactly as a dictionary with these keys:
{
    'Airline': 'easyJet',
    'Flight Number': 'U2 4631',
    'Departure Airport': 'Paris Charles de Gaulle Airport (CDG)',
    'Departure Time': 'October 10, 2025, at 07:25',
    'Arrival Airport': 'Berlin Brandenburg Airport (BER)',
    'Arrival Time': 'October 10, 2025, at 09:10',
    'Duration': '1 hour and 45 minutes',
    'Aircraft': 'Airbus A320',
    'Travel Class': 'Economy',
    'Legroom': '29 inches',
    'Price': '$122'
}

Do not add any additional text or commentary around the dictionary. Return only the required dictionary output formatted as shown above.
"""


In [ ]:
# @title --- Flight Agent ---
@tool
def search_flights(origin: str, destination: str, date: str, trip_type : int) -> str:
    """
    Fetches the flight details from google

    Args:
    origin (str): from where you want to take the flight
    destination (str): where you want to go
    date  (str): on which date want to take the flight
    trip_type (int): type of flight is it a one-way trip or a round trip, choose the options appropriately {"Round trip":1, "One way":2, "Multi-city":3}
    """

    params = {
            "api_key": serp_api,
            "engine": "google_flights",
            "hl": "en",
            "gl": "us",
            "departure_id": origin,
            "arrival_id": destination,
            "outbound_date": date,
            "type": trip_type,
            "currency": "USD"
    }

    results = GoogleSearch(params).get_dict()
    return results

flight_agent = CodeAgent(tools=[search_flights], model=model, name="flight_agent", description="Searches for flight details between two locations for a given date.", instructions=flight_system_prompt )

### **Define - Hotel Tool**

In [ ]:
hotel_system_prompt = """
You are a Travel Assistant Agent responsible for providing hotel information for nearby accommodations with key details.

Return a dictionary with the following keys and example values:
{
    "Hotel Name": "Hyatt Place Frankfurt Airport",
    "Star Rating": "5 Stars",
    "Price Per Night": "$95",
    "Distance from Airport": "15 min taxi",
    "Amenities": "Free Wi-Fi, Gym, Restaurant, Bar, Room service"
}

Do not add any additional text or commentary around the dictionary. Return only the required dictionary output formatted as shown above.
"""

In [ ]:
# @title --- Hotel Agent ---
@tool
def search_hotel(location: str, check_in_date: str, check_out_date: str) -> str:
    """
    Fetches the hotel details from google

    Args:
    location (str): Location where you want to search the hotels
    check_in_date (str): At what date do you want to check in
    check_out_date  (str): At what date do you want to check out
    """

    params = {
            "api_key": serp_api,
            "engine": "google_hotels",
            "q": location,
            "check_in_date": check_in_date,
            "check_out_date": check_out_date,
            "currency": "USD"
    }

    hotel_results = GoogleSearch(params).get_dict()
    return hotel_results

hotel_agent = CodeAgent(tools=[search_hotel], model=model, name="hotel_agent", description="Searches for hotels in the desired drop location of the flight", instructions=hotel_system_prompt )

## **Manger Agent**

In [ ]:
# @title --- Manager Agent orchestrates the workflow ---
manager_agent = CodeAgent(
    model=model,
    managed_agents=[flight_agent, hotel_agent],
    tools=[WebSearchTool()]
)

## **Agent Call**

# @title --- Example user query/task ---
task = """
Find me 3 flight options from Paris to Berlin, Germany that depart within the next 2 days,
covering a range from the cheapest available flight to the most expensive nonstop flight.
For each flight, provide airline, departure time, arrival time, duration, number of stops, and price.
Also, find 3 accommodation options for a 2-night stay in Berlin at varying distances from the
city-center business district, including hotel name, star rating, price per night, distance, and amenities.
"""

# Guarded execution: call the live manager agent only when we have keys AND opted in.
# Otherwise fall back to the cached dataset so the notebook is fully reproducible.
if RUN_LIVE and model is not None:
    result = manager_agent.run(task)
else:
    print("[info] RUN_LIVE is False -> using CACHED_RESULT (no API calls made).")
    result = CACHED_RESULT

result


In [ ]:
# @title --- Cached search results (used when RUN_LIVE is False) ---
# ---------------------------------------------------------------------------
# WHY THIS EXISTS
# The flight/hotel agents call live SerpAPI + an LLM, so results change every
# run and require paid keys. To make this notebook REPRODUCIBLE and runnable
# with no credentials, we cache one representative response here. All evaluation
# below runs against this dataset, so the graded numbers are deterministic.
#
# The data is deliberately shaped to expose the assignment's core lesson:
# the cheapest LISTED option is not the cheapest TOTAL-BUSINESS-COST option.
#   Flights: A is cheapest airfare but has 2 stops / 6h; C is priciest but nonstop.
#   Hotels:  X is cheapest/night but a long commute; Y is priciest but central.
#
# Fields are intentionally MESSY (prices as strings, durations in natural
# language, commute as mixed units) to exercise the robust parsers in Part 2.
# ---------------------------------------------------------------------------
CACHED_RESULT = {
    "flights": {
        "Option A": {
            "Airline": "BudgetWings", "Flight Number": "BW 88",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "6 hours", "Stops": 2, "Price": "$130",
        },
        "Option B": {
            "Airline": "EuroConnect", "Flight Number": "EC 210",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "3 hours and 30 minutes", "Stops": 1, "Price": "$210",
        },
        "Option C": {
            "Airline": "Lufthansa", "Flight Number": "LH 1032",
            "Departure Airport": "Paris Charles de Gaulle (CDG)",
            "Arrival Airport": "Berlin Brandenburg (BER)",
            "Duration": "1 hour and 50 minutes", "Stops": 0, "Price": "$340",
        },
    },
    "hotels": {
        "Hotels Found": [
            {
                "Hotel Name": "BER Airport Area North Inn", "Star Rating": "4.2 Stars",
                "Price Per Night": "$94", "Distance from Airport": "35 min taxi",
                "Commute Time": "90 minutes",  # long daily commute to the city center
                "Amenities": "Free Wi-Fi, Restaurant",
            },
            {
                "Hotel Name": "Berlin Mitte Comfort", "Star Rating": "4.0 Stars",
                "Price Per Night": "$140", "Distance from Airport": "20 min taxi",
                "Commute Time": "0.8 hours",  # moderate commute, mixed unit format
                "Amenities": "Free Wi-Fi, Gym, Bar",
            },
            {
                "Hotel Name": "City Center Grand", "Star Rating": "5 Stars",
                "Price Per Night": "$200", "Distance from Airport": "12 min taxi",
                "Commute Time": "18 min",  # central: short commute to business site
                "Amenities": "Spa, Restaurant, Bar, Gym",
            },
        ]
    },
}

print("[info] Cached dataset ready:",
      len(CACHED_RESULT["flights"]), "flights,",
      len(CACHED_RESULT["hotels"]["Hotels Found"]), "hotels.")


In [ ]:
# @title --- Example user query/task ---
task = """
Find me 2 flight options from Paris to any destination in Germany that depart within the next 2 days: one being the cheapest available flight and the other the most expensive flight. For each flight, please provide details including airline, departure time, arrival time, duration, number of stops, and price.
Also, find accommodation options for a 2-day stay in Germany near the arrival airport. Please provide options for luxurious 5-star hotels within 15 km of the airport, including hotel name, star rating, price per night, distance from the airport, and available amenities.
"""

result = manager_agent.run(task)
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find me 2 flight options from Paris to any destination in Germany that depart within the next 2 days: one being │
│ the cheapest available flight and the other the most expensive flight. For each flight, please provide details  │
│ including airline, departure time, arrival time, duration, number of stops, and price.                          │
│ Also, find accommodation options for a 2-day stay in Germany near the arrival airport. Please provide options   │
│ for luxurious 5-star hotels within 15 km of the airport, including hotel name, star rating, price per night,    │
│ distance from the airport, and available amenities.                                                             │
│                                                                                                                 │
╰─ OpenAIServerModel - gpt-4o-mini ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  flight_details = flight_agent(task="Find 2 flight options from Paris to any destination in Germany that depart   
  within the next 2 days: one being the cheapest available flight and the other the most expensive flight.",       
  additional_args={})                                                                                              
  print(flight_details)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭──────────────────────────────────────────── New run - flight_agent ─────────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'flight_agent'.                                                                    │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Find 2 flight options from Paris to any destination in Germany that depart within the next 2 days: one being    │
│ the cheapest available flight and the other the most expensive flight.                                          │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│                                                                                                                 │
╰─ OpenAIServerModel - gpt-4o-mini ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  flight_summary = {                                                                                               
      "Cheapest Flight": {                                                                                         
          "Airline": "easyJet",                                                                                    
          "Flight Number": "U2 4631",                                                                              
          "Departure Time": "2025-10-10 07:25",                                                                    
          "Arrival Airport": "Berlin Brandenburg Airport",                                                         
          "Arrival Time": "2025-10-10 09:10",                                                                      
          "Duration": "1 hour and 45 minutes",                                                                     
          "Price": "$122"                                                                                          
      },                                                                                                           
      "Most Expensive Flight": {                                                                                   
          "Airline": "Scandinavian Airlines",                                                                      
          "Flight Number": "SK 564",                                                                               
          "Departure Time": "2025-10-10 10:10",                                                                    
          "Arrival Airport": "Copenhagen Airport",                                                                 
          "Arrival Time": "2025-10-10 12:05",                                                                      
          "Duration": "3 hours and 40 minutes",                                                                    
          "Price": "$276"                                                                                          
      }                                                                                                            
  }                                                                                                                
                                                                                                                   
  hotel_summary = {                                                                                                
      "Hotels Found": [                                                                                            
          {                                                                                                        
              "Hotel Name": "IntercityHotel Berlin Airport BER Terminal 1+2",                                      
              "Star Rating": "4.4 Stars",                                                                          
              "Price Per Night": "$291",                                                                           
              "Distance from Airport": "3 min taxi",                                                               
              "Amenities": "Restaurant, Bar, Exercise room"                                                        
          },                                                                                                       
          {                                                                                                        
              "Hotel Name": "Steigenberger Airport Hotel Berlin",                                                  
              "Star Rating": "4.2 Stars",                                                                          
              "Price Per Night": "$359",                 

Final answer: {'flights': {'Cheapest Flight': {'Airline': 'easyJet', 'Flight Number': 'U2 4631', 'Departure Time': 
'2025-10-10 07:25', 'Arrival Airport': 'Berlin Brandenburg Airport', 'Arrival Time': '2025-10-10 09:10', 
'Duration': '1 hour and 45 minutes', 'Price': '$122'}, 'Most Expensive Flight': {'Airline': 'Scandinavian 
Airlines', 'Flight Number': 'SK 564', 'Departure Time': '2025-10-10 10:10', 'Arrival Airport': 'Copenhagen 
Airport', 'Arrival Time': '2025-10-10 12:05', 'Duration': '3 hours and 40 minutes', 'Price': '$276'}}, 'hotels': 
{'Hotels Found': [{'Hotel Name': 'IntercityHotel Berlin Airport BER Terminal 1+2', 'Star Rating': '4.4 Stars', 
'Price Per Night': '$291', 'Distance from Airport': '3 min taxi', 'Amenities': 'Restaurant, Bar, Exercise room'}, 
{'Hotel Name': 'Steigenberger Airport Hotel Berlin', 'Star Rating': '4.2 Stars', 'Price Per Night': '$359', 
'Distance from Airport': '6 min taxi', 'Amenities': 'Spa, Pet-friendly, Kids-friendly, Restaurant, Bar'}, {'Hotel 
Name': 'IntercityHotel Berlin Airport Area North', 'Star Rating': '4.2 Stars', 'Price Per Night': '$94', 'Distance 
from Airport': '7 min taxi', 'Amenities': 'Free Wi-Fi, Restaurant, Bar'}]}}

[Step 4: Duration 17.14 seconds| Input tokens: 14,463 | Output tokens: 2,021]

{'flights': {'Cheapest Flight': {'Airline': 'easyJet', 'Flight Number': 'U2 4631', 'Departure Time': '2025-10-10 07:25', 'Arrival Airport': 'Berlin Brandenburg Airport', 'Arrival Time': '2025-10-10 09:10', 'Duration': '1 hour and 45 minutes', 'Price': '$122'}, 'Most Expensive Flight': {'Airline': 'Scandinavian Airlines', 'Flight Number': 'SK 564', 'Departure Time': '2025-10-10 10:10', 'Arrival Airport': 'Copenhagen Airport', 'Arrival Time': '2025-10-10 12:05', 'Duration': '3 hours and 40 minutes', 'Price': '$276'}}, 'hotels': {'Hotels Found': [{'Hotel Name': 'IntercityHotel Berlin Airport BER Terminal 1+2', 'Star Rating': '4.4 Stars', 'Price Per Night': '$291', 'Distance from Airport': '3 min taxi', 'Amenities': 'Restaurant, Bar, Exercise room'}, {'Hotel Name': 'Steigenberger Airport Hotel Berlin', 'Star Rating': '4.2 Stars', 'Price Per Night': '$359', 'Distance from Airport': '6 min taxi', 'Amenities': 'Spa, Pet-friendly, Kids-friendly, Restaurant, Bar'}, {'Hotel Name': 'IntercityHote

In [ ]:
result

{'flights': {'Cheapest Flight': {'Airline': 'easyJet',
   'Flight Number': 'U2 4631',
   'Departure Time': '2025-10-10 07:25',
   'Arrival Airport': 'Berlin Brandenburg Airport',
   'Arrival Time': '2025-10-10 09:10',
   'Duration': '1 hour and 45 minutes',
   'Price': '$122'},
  'Most Expensive Flight': {'Airline': 'Scandinavian Airlines',
   'Flight Number': 'SK 564',
   'Departure Time': '2025-10-10 10:10',
   'Arrival Airport': 'Copenhagen Airport',
   'Arrival Time': '2025-10-10 12:05',
   'Duration': '3 hours and 40 minutes',
   'Price': '$276'}},
 'hotels': {'Hotels Found': [{'Hotel Name': 'IntercityHotel Berlin Airport BER Terminal 1+2',
    'Star Rating': '4.4 Stars',
    'Price Per Night': '$291',
    'Distance from Airport': '3 min taxi',
    'Amenities': 'Restaurant, Bar, Exercise room'},
   {'Hotel Name': 'Steigenberger Airport Hotel Berlin',
    'Star Rating': '4.2 Stars',
    'Price Per Night': '$359',
    'Distance from Airport': '6 min taxi',
    'Amenities': 'Spa, Pet

## **Goal Check**

1. Our objective is to verify whether the selected flights fall within the specified budget.

2. We need to confirm whether our goal is being met - is the agent accurately determining this?

In [ ]:
# Original (price-only) goal-based flight evaluation.
budget_goal = 200.0     # USD airfare budget used by the original workflow
penalty_factor = 0.1    # penalty applied to over-budget airfare

evaluation_task = f"""
                  Your task is to check if the below provided flight details are within the budget or not
                  and return the penalty scores accordingly.
                  The budget for the flights is {budget_goal} and the penalty is {penalty_factor} for going above the budget.
                  Here are the Results : {result['flights']}
                  """

# Guarded: use the LLM agent live; offline, apply the ORIGINAL goal_checker tool
# directly to each flight so the price-only baseline is reproduced deterministically.
if RUN_LIVE and model is not None:
    goal_based_agent_response = goal_checker_agent.run(evaluation_task)
else:
    print("[info] Offline -> applying the original goal_checker tool to each flight.")
    goal_based_agent_response = {
        name: goal_checker(flight, budget_goal, penalty_factor)
        for name, flight in result["flights"].items()
    }

goal_based_agent_response


In [ ]:
# @title --- Goal Based Agent - Flight Evaluator ---
@tool
def goal_checker(flight: Dict, budget_goal: float, penalty_factor: float) -> float:
    """
    Evaluate the flight based on budget goal

    Args:
    flight (dict): Contains the flight details like Price, Origin, Destination, Time, Date,...etc
    budget_goal (float): What is the budget, to be used for scoring the flight, the budget is in the USD
    penalty_factor (float): How to penalize the score if the flight is over budget
    """
    if isinstance(flight['Price'], str):
        price = float(flight['Price'].replace('$', ''))
    else:
        price = flight['Price']

    if price <= budget_goal:
        savings = budget_goal - price
        return {
            "Price":price,
            "Score": round(savings * (1.0 - penalty_factor), 2),
            "Savings": round(savings, 2)
            }

    else:
        excess = price - budget_goal
        return {
            "Price":price,
            "Score": round(-excess * penalty_factor, 2),
            "Difference": round(excess, 2)
        }

goal_checker_agent = CodeAgent(tools=[goal_checker], model=model, name="goal_checker_agent", description="Evaluate the flight based on budget goal", instructions=goal_check_system_prompt)

In [ ]:
evaluation_task = f"""
                  Your task is to the check, if the below provided flight details are within the budget or not and return the penalty scores accordingly.
                  The budget for the flights is 200 and the penalty is 0.1 for going above the budget
                  Here are the Results : {result['flights']}
                  """

goal_based_agent_response = goal_checker_agent.run(evaluation_task)

╭───────────────────────────────────────── New run - goal_checker_agent ──────────────────────────────────────────╮
│                                                                                                                 │
│ Your task is to the check, if the below provided flight details are within the budget or not and return the     │
│ penalty scores accordingly.                                                                                     │
│                   The budget for the flights is 200 and the penalty is 0.1 for going above the budget           │
│                   Here are the Results : {'Cheapest Flight': {'Airline': 'easyJet', 'Flight Number': 'U2 4631', │
│ 'Departure Time': '2025-10-10 07:25', 'Arrival Airport': 'Berlin Brandenburg Airport', 'Arrival Time':          │
│ '2025-10-10 09:10', 'Duration': '1 hour and 45 minutes', 'Price': '$122'}, 'Most Expensive Flight': {'Airline': │
│ 'Scandinavian Airlines', 'Flight Number': 'SK 564', 'Departure Time': '2025-10-10 10:10', 'Arrival Airport':    │
│ 'Copenhagen Airport', 'Arrival Time': '2025-10-10 12:05', 'Duration': '3 hours and 40 minutes', 'Price':        │
│ '$276'}}                                                                                                        │
│                                                                                                                 │
╰─ OpenAIServerModel - gpt-4o-mini ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Defining flight details                                                                                        
  flights = {                                                                                                      
      'Cheapest Flight': {'Price': '$122'},                                                                        
      'Most Expensive Flight': {'Price': '$276'}                                                                   
  }                                                                                                                
  budget_goal = 200                                                                                                
  penalty_factor = 0.1                                                                                             
                                                                                                                   
  # Process the flights                                                                                            
  results = {}                                                                                                     
  for flight_type, details in flights.items():                                                                     
      price = float(details['Price'].replace('$', ''))                                                             
      score = goal_checker(flight=details, budget_goal=budget_goal, penalty_factor=penalty_factor)                 
      status = "Within Budget" if price <= budget_goal else "Over Budget"                                          
      results[flight_type] = {'Price': price, 'Score': score, 'Status': status}                                    
                                                                                                                   
  results                                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: {'Cheapest Flight': {'Price': 122.0, 'Score': {'Price': 122.0, 'Score': 70.2, 'Savings': 78.0}, 'Status': 
'Within Budget'}, 'Most Expensive Flight': {'Price': 276.0, 'Score': {'Price': 276.0, 'Score': -7.6, 'Difference': 
76.0}, 'Status': 'Over Budget'}}

[Step 1: Duration 4.77 seconds| Input tokens: 2,477 | Output tokens: 233]

...

[Step 2: Duration 4.53 seconds| Input tokens: 5,513 | Output tokens: 430]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({                                                                                                   
      'Cheapest Flight': {'Price': 122.0, 'Score': 70.2, 'Status': 'Within Budget'},                               
      'Most Expensive Flight': {'Price': 276.0, 'Score': -7.6, 'Status': 'Over Budget'}                            
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'Cheapest Flight': {'Price': 122.0, 'Score': 70.2, 'Status': 'Within Budget'}, 'Most Expensive 
Flight': {'Price': 276.0, 'Score': -7.6, 'Status': 'Over Budget'}}

[Step 3: Duration 2.18 seconds| Input tokens: 9,007 | Output tokens: 499]

In [ ]:
goal_based_agent_response

{'Cheapest Flight': {'Price': 122.0, 'Score': 70.2, 'Status': 'Within Budget'},
 'Most Expensive Flight': {'Price': 276.0,
  'Score': -7.6,
  'Status': 'Over Budget'}}

## **Utility Check**

Here, we are evaluating the score based on two criteria:

1. The hotel's price (or price per night) must fall within our budget.

2. The hotel must have a 5-star rating.

In [ ]:
# Original (price + star-rating only) utility-based hotel evaluation.
hotel_budget = 100.0   # USD per-night budget used by the original workflow

evaluation_task = f"""
                  Your task is to evaluate all the hotels individually, checking if the below provided hotel details
                  are within the budget or not and return the utility scores accordingly.
                  The budget for the hotel is ${hotel_budget}.
                  Here are the Results : {result['hotels']}
                  """

# Guarded: use the LLM agent live; offline, apply the ORIGINAL utility_checker tool
# directly to each hotel so the price+rating baseline is reproduced deterministically.
if RUN_LIVE and model is not None:
    utility_checker_agent_response = utility_checker_agent.run(evaluation_task)
else:
    print("[info] Offline -> applying the original utility_checker tool to each hotel.")
    utility_checker_agent_response = [
        {"Hotel": h.get("Hotel Name"), "utility_score": utility_checker(h, hotel_budget)}
        for h in result["hotels"]["Hotels Found"]
    ]

utility_checker_agent_response


In [ ]:
# @title --- Utility Based Agent - Hotel Evaluator ---

from typing import Dict

@tool
def utility_checker(hotel: Dict, budget: float) -> Dict:
    """
    Evaluate the hotel based on star rating and budget to compute a utility score.

    Args:
        hotel (dict): Contains hotel details like 'Price Per Night', 'Star Rating', etc.
        budget (float): Budget to be used for scoring the hotel in USD.

    Returns:
        dict: Utility evaluation containing cumulative utility_score, parsed price, and scoring breakdown.
    """

    utility_score = 0

    # --- Star rating check ---
    stars_raw = hotel.get('Rating', 0) or hotel.get('Star Rating', 0)
    if ("5" in stars_raw) or (stars_raw == 5):
        utility_score += 50  # bonus for desired rating
    else:
        utility_score -= 50  # penalty for lower rating


    # --- Price check ---
    price_raw = hotel.get('Price Per Night', '0') or hotel.get('Price', '0') or hotel.get('Price per Night', '0')
    if isinstance(price_raw, str):
        price = float(price_raw.replace('$', '').replace(',', '').strip())
    else:
        price = float(price_raw)

    if price <= budget:
        utility_score += 50  # reward for meeting budget
    else:
        utility_score -= 50  # penalty for exceeding budget

    return utility_score


utility_checker_agent = CodeAgent(tools=[utility_checker], model=model, name="utility_checker_agent", description="Evaluate the hotels based on star rating and budget", instructions=utility_checker_system_prompt )

In [ ]:
evaluation_task = f"""
                  Your task is to the evaluiate all the hotels individually, if the below provided hotel details are within the budget or not and return the utility scores accordingly.
                  The budget for the hotel is $100.
                  Here are the Results : {result['hotels']}
                  """

utility_checker_agent_response = utility_checker_agent.run(evaluation_task)

╭──────────────────────────────────────── New run - utility_checker_agent ────────────────────────────────────────╮
│                                                                                                                 │
│ Your task is to the evaluiate all the hotels individually, if the below provided hotel details are within the   │
│ budget or not and return the utility scores accordingly.                                                        │
│                   The budget for the hotel is $100.                                                             │
│                   Here are the Results : {'Hotels Found': [{'Hotel Name': 'IntercityHotel Berlin Airport BER    │
│ Terminal 1+2', 'Star Rating': '4.4 Stars', 'Price Per Night': '$291', 'Distance from Airport': '3 min taxi',    │
│ 'Amenities': 'Restaurant, Bar, Exercise room'}, {'Hotel Name': 'Steigenberger Airport Hotel Berlin', 'Star      │
│ Rating': '4.2 Stars', 'Price Per Night': '$359', 'Distance from Airport': '6 min taxi', 'Amenities': 'Spa,      │
│ Pet-friendly, Kids-friendly, Restaurant, Bar'}, {'Hotel Name': 'IntercityHotel Berlin Airport Area North',      │
│ 'Star Rating': '4.2 Stars', 'Price Per Night': '$94', 'Distance from Airport': '7 min taxi', 'Amenities': 'Free │
│ Wi-Fi, Restaurant, Bar'}\]}                                                                                     │
│                                                                                                                 │
╰─ OpenAIServerModel - gpt-4o-mini ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  hotels = [                                                                                                       
      {'Hotel Name': 'IntercityHotel Berlin Airport BER Terminal 1+2', 'Star Rating': '4.4 Stars', 'Price Per      
  Night': '$291'},                                                                                                 
      {'Hotel Name': 'Steigenberger Airport Hotel Berlin', 'Star Rating': '4.2 Stars', 'Price Per Night':          
  '$359'},                                                                                                         
      {'Hotel Name': 'IntercityHotel Berlin Airport Area North', 'Star Rating': '4.2 Stars', 'Price Per Night':    
  '$94'}                                                                                                           
  ]                                                                                                                
                                                                                                                   
  budget = 100                                                                                                     
  output = []                                                                                                      
                                                                                                                   
  for hotel in hotels:                                                                                             
      price = int(hotel['Price Per Night'].replace('$', ''))                                                       
      if price <= budget:                                                                                          
          utility_score = 100                                                                                      
      else:                                                                                                        
          utility_score = 0                                                                                        
                                                                                                                   
      output.append({'Hotel': hotel['Hotel Name'], 'utility_score': utility_score})                                
                                                                                                                   
  print(output)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[{'Hotel': 'IntercityHotel Berlin Airport BER Terminal 1+2', 'utility_score': 0}, {'Hotel': 'Steigenberger Airport 
Hotel Berlin', 'utility_score': 0}, {'Hotel': 'IntercityHotel Berlin Airport Area North', 'utility_score': 100}]

Out: None

[Step 1: Duration 6.30 seconds| Input tokens: 2,498 | Output tokens: 296]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_output = [                                                                                                 
      {'Hotel': 'IntercityHotel Berlin Airport BER Terminal 1+2', 'utility_score': 0},                             
      {'Hotel': 'Steigenberger Airport Hotel Berlin', 'utility_score': 0},                                         
      {'Hotel': 'IntercityHotel Berlin Airport Area North', 'utility_score': 100}                                  
  ]                                                                                                                
                                                                                                                   
  final_answer(final_output)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: [{'Hotel': 'IntercityHotel Berlin Airport BER Terminal 1+2', 'utility_score': 0}, {'Hotel': 
'Steigenberger Airport Hotel Berlin', 'utility_score': 0}, {'Hotel': 'IntercityHotel Berlin Airport Area North', 
'utility_score': 100}]

[Step 2: Duration 3.61 seconds| Input tokens: 5,607 | Output tokens: 471]

In [ ]:
print(utility_checker_agent_response)

[{'Hotel': 'IntercityHotel Berlin Airport BER Terminal 1+2', 'utility_score': 0}, {'Hotel': 'Steigenberger Airport Hotel Berlin', 'utility_score': 0}, {'Hotel': 'IntercityHotel Berlin Airport Area North', 'utility_score': 100}]
